# Etapa 3 — Reproducir las métricas del paper (sin llamar a ninguna API)

Paper: *Benchmarking biochemical networks generated by large language models* (eLife 109709).
Repo original: <https://github.com/saucermanlab/LLM-network-generation> (licencia MIT, hay que citarlo).

**Qué hacemos aquí:** tomamos las 10 respuestas guardadas de cada LLM (GPT-5.2 Pro, Gemini 3 Pro, Claude Opus 4.6)
para las 3 redes (hipertrofia, fibroblasto, mecanoseñalización), recalculamos **recall** y **precisión**
y comparamos con lo que el repo ya trae calculado.

- **Recall** = de las flechas que existen en la red hecha a mano, ¿qué fracción encontró el LLM? `TP / (TP + FN)`
- **Precisión** = de las flechas que propuso el LLM, ¿qué fracción existe en la red hecha a mano? `TP / (TP + FP)`
- Una flecha cuenta como acierto solo si coinciden **origen, destino y signo** (activa/inhibe).

Se importan del repo `get_model_data`, `extract_AND_reactions` y `compare_connections`. Las funciones de recall/precisión viven dentro
de `figure_generation.ipynb` del repo (no son importables), así que están copiadas en `evaluacion.py` (ver su cabecera).

In [1]:
import sys
sys.dont_write_bytecode = True          # no crear __pycache__ dentro del repo
import numpy as np, pandas as pd
from pathlib import Path
import evaluacion as ev                  # nuestro módulo (importa las funciones del repo)

REDES = ["hypertrophy", "fibroblast", "mechanosignaling"]
LLMS  = ["claude", "gpt", "gemini"]
TABLAS = ev.REPO / "outputs" / "prediction_evaluation_tables"
SALIDA = Path(ev.RAIZ / "experimento" / "resultados"); SALIDA.mkdir(exist_ok=True)
print("Python:", sys.version.split()[0], "|", sys.executable)
print("Repo:", ev.REPO)

Python: 3.11.9 | C:\Users\lauta\Desktop\Biologia Computacional\Examen Parcial\Benchmarking biochemical networks\.venv\Scripts\python.exe
Repo: C:\Users\lauta\Desktop\Biologia Computacional\Examen Parcial\Benchmarking biochemical networks\LLM-network-generation


## Parte A — Conteo de flechas verdaderas halladas (con `compare_connections` del repo)

`compare_connections` devuelve una fila por cada flecha de la red hecha a mano y una columna (1..10) por réplica con 1 si el LLM la propuso.
Comparamos de dos formas con lo que trae el repo:
1. ¿La tabla que recalculamos es **idéntica** a `manuscript_tables/`?
2. ¿Los conteos por réplica coinciden con `recall_tables/`? (es el "recall" de la Figura 1, expresado como número de flechas, no como %)

In [2]:
filas = []
tablas_A = {}
for red in REDES:
    saved = pd.read_excel(TABLAS / "recall_tables" / f"recall_{red}.xlsx")
    for llm in LLMS:
        t = ev.tabla_aciertos(red, llm)
        tablas_A[(red, llm)] = t
        # 1) ¿igual a la tabla guardada del manuscrito?
        g = pd.read_excel(TABLAS / "manuscript_tables" / f"predicted_connections_{red}_{llm}.xlsx")
        cols = ['source','target','action','AND','ANDindex'] + [str(i) for i in range(1, 11)]
        iguales = t[cols].reset_index(drop=True).equals(g[cols].reset_index(drop=True))
        # 2) conteos por réplica vs recall_tables
        conteos, n_verdad = ev.conteo_aciertos(t)
        mios = [conteos[str(i)] for i in range(1, 11)]
        guardado = saved[llm.capitalize() if llm != 'gpt' else 'GPT'].tolist()
        reps = [str(i) for i in range(1, 11)]
        dist = int((t[reps].reset_index(drop=True).values != g[reps].reset_index(drop=True).values).sum())
        filas.append({"red": red, "llm": llm, "flechas_verdad": n_verdad,
                      "tabla_identica_a_manuscrito": iguales, "celdas_distintas": dist,
                      "conteos_coinciden_recall_tables": mios == guardado,
                      "media_aciertos": np.mean(mios), "sd": np.std(mios, ddof=1)})
resumen_A = pd.DataFrame(filas)
resumen_A.round(2)

,red,llm,flechas_verdad,tabla_identica_a_manuscrito,celdas_distintas,conteos_coinciden_recall_tables,media_aciertos,sd
0,hypertrophy,claude,191,True,0,True,119.0,2.31
1,hypertrophy,gpt,191,True,0,True,107.5,3.84
2,hypertrophy,gemini,191,True,0,True,74.5,3.24
3,fibroblast,claude,162,True,0,True,95.6,2.12
4,fibroblast,gpt,162,True,0,True,73.6,5.04
5,fibroblast,gemini,162,True,0,True,45.5,3.21
6,mechanosignaling,claude,172,False,11,False,107.0,2.31
7,mechanosignaling,gpt,172,False,10,False,82.2,6.84
8,mechanosignaling,gemini,172,False,10,False,56.2,7.10


**Resultado de la Parte A.** Hipertrofia y fibroblasto: la tabla recalculada es idéntica a la guardada y los conteos coinciden con `recall_tables/`.
**Mecanoseñalización: NO coincide del todo**, y es por una diferencia del propio repo, no de nuestro cálculo:
la tabla guardada marca un acierto extra por réplica en la flecha `AngII → AT1R`. La causa es que en esa red el segundo nodo que comparte el gen `AGT`
se llama `Ao` (no `AGT`), y el código actual de `compare_connections` solo duplica la flecha para `AngII` si ese nodo se llama `AGT`.
Es decir, el código actual del repo ya no reproduce esa tabla guardada. Esto no afecta a nuestro experimento (solo usamos hipertrofia) y las métricas formales de la Parte B sí coinciden.

## Parte B — Recall y precisión con matrices nodo × nodo (el cálculo "formal" del paper)

Se construye una matriz con 1 (estimula), −1 (inhibe) o 0 (sin flecha) para la red verdadera y para cada réplica, y de ahí salen TP, FP, FN, TN.
Calculamos las métricas por réplica y luego promediamos. Los valores de referencia están en el repo: `outputs/figures/confusion_matrices/summary.xlsx`.

In [3]:
filas_rep = []
for red in REDES:
    gt, vacia = ev.matriz_verdad(red)
    for llm in LLMS:
        for idx, df in sorted(ev.cargar_jsons_manuscrito(red, llm).items()):
            m = ev.metricas(ev.matriz_prediccion(df, red, vacia), gt)
            filas_rep.append({"red": red, "llm": llm, "replica": idx + 1, **m})
por_replica = pd.DataFrame(filas_rep)
por_replica.groupby(["red", "llm"]).size().unstack()   # debe haber 10 réplicas en cada celda

llm,claude,gemini,gpt
red,,,
fibroblast,10,10,10
hypertrophy,10,10,10
mechanosignaling,10,10,10


In [4]:
ref = pd.read_excel(ev.REPO / "outputs" / "figures" / "confusion_matrices" / "summary.xlsx")
ref = ref.rename(columns={"model": "red", "llm": "llm", "Sensitivity": "recall_repo", "Precision": "precision_repo",
                          "TP": "TP_repo", "FP": "FP_repo", "FN": "FN_repo"})
agg = por_replica.groupby(["red", "llm"]).agg(
    n_replicas=("replica", "count"),
    TP=("TP", "mean"), FP=("FP", "mean"), FN=("FN", "mean"),
    recall_media=("recall", "mean"), recall_sd=("recall", lambda s: s.std(ddof=1)),
    precision_media=("precision", "mean"), precision_sd=("precision", lambda s: s.std(ddof=1)),
    verdad_total=("verdad_total", "first"), signo_contrario_media=("signo_contrario", "mean"),
).reset_index()
comp = agg.merge(ref[["red", "llm", "TP_repo", "FP_repo", "FN_repo", "recall_repo", "precision_repo"]], on=["red", "llm"])
comp["dif_recall"] = (comp["recall_media"] - comp["recall_repo"]).abs()
comp["dif_precision"] = (comp["precision_media"] - comp["precision_repo"]).abs()
comp["coincide"] = (comp["dif_recall"] < 1e-9) & (comp["dif_precision"] < 1e-9)
comp.round(3)

,red,llm,n_replicas,TP,FP,FN,recall_media,recall_sd,precision_media,precision_sd,verdad_total,signo_contrario_media,TP_repo,FP_repo,FN_repo,recall_repo,precision_repo,dif_recall,dif_precision,coincide
0,fibroblast,claude,10,94.2,201.6,65.2,0.591,0.014,0.319,0.013,162,2.6,94.2,201.6,65.2,0.591,0.319,0.0,0.0,True
1,fibroblast,gemini,10,45.5,17.9,116.5,0.281,0.020,0.717,0.032,162,0.0,45.5,17.9,116.5,0.281,0.717,0.0,0.0,True
2,fibroblast,gpt,10,72.1,141.4,87.3,0.452,0.033,0.342,0.035,162,2.6,72.1,141.4,87.3,0.452,0.342,0.0,0.0,True
3,hypertrophy,claude,10,119.0,103.0,69.4,0.632,0.013,0.537,0.025,191,2.6,119.0,103.0,69.4,0.632,0.537,0.0,0.0,True
4,hypertrophy,gemini,10,74.5,24.9,116.5,0.390,0.017,0.751,0.034,191,0.0,74.5,24.9,116.5,0.390,0.751,0.0,0.0,True
5,hypertrophy,gpt,10,107.5,124.7,82.4,0.566,0.021,0.464,0.029,191,1.1,107.5,124.7,82.4,0.566,0.464,0.0,0.0,True
6,mechanosignaling,claude,10,106.4,215.8,59.3,0.642,0.016,0.332,0.021,172,6.3,106.4,215.8,59.3,0.642,0.332,0.0,0.0,True
7,mechanosignaling,gemini,10,56.2,21.6,115.2,0.328,0.041,0.729,0.047,172,0.6,56.2,21.6,115.2,0.328,0.729,0.0,0.0,True
8,mechanosignaling,gpt,10,81.5,173.7,87.6,0.482,0.041,0.330,0.057,172,2.9,81.5,173.7,87.6,0.482,0.330,0.0,0.0,True


In [5]:
print("¿Todas las combinaciones coinciden con summary.xlsx del repo?", bool(comp["coincide"].all()))
ok_A = resumen_A[resumen_A["red"] != "mechanosignaling"]
print("¿Parte A coincide con manuscript_tables y recall_tables en hipertrofia y fibroblasto?",
      bool(ok_A["tabla_identica_a_manuscrito"].all() and ok_A["conteos_coinciden_recall_tables"].all()))
print("Mecanoseñalización: celdas distintas en la tabla (explicado arriba):",
      resumen_A[resumen_A["red"] == "mechanosignaling"].set_index("llm")["celdas_distintas"].to_dict())

¿Todas las combinaciones coinciden con summary.xlsx del repo? True
¿Parte A coincide con manuscript_tables y recall_tables en hipertrofia y fibroblasto? True
Mecanoseñalización: celdas distintas en la tabla (explicado arriba): {'claude': 11, 'gpt': 10, 'gemini': 10}


## Un detalle del método (lo explicamos en el reporte)

Cuando el LLM propone una flecha real pero con el **signo contrario**, el repo la cuenta como falso positivo, pero **no** como falso negativo.
Entonces `TP + FN` es menor que el total de flechas verdaderas y el recall sale un poco más alto de lo que sería si se contara solo `TP / total`.
Medimos cuánto cambia:

In [6]:
comp["recall_estricto"] = comp["TP"] / comp["verdad_total"]      # TP / todas las flechas de la red hecha a mano
comp[["red", "llm", "verdad_total", "TP", "FN", "signo_contrario_media", "recall_media", "recall_estricto"]].round(3)

,red,llm,verdad_total,TP,FN,signo_contrario_media,recall_media,recall_estricto
0,fibroblast,claude,162,94.2,65.2,2.6,0.591,0.581
1,fibroblast,gemini,162,45.5,116.5,0.0,0.281,0.281
2,fibroblast,gpt,162,72.1,87.3,2.6,0.452,0.445
3,hypertrophy,claude,191,119.0,69.4,2.6,0.632,0.623
4,hypertrophy,gemini,191,74.5,116.5,0.0,0.390,0.390
5,hypertrophy,gpt,191,107.5,82.4,1.1,0.566,0.563
6,mechanosignaling,claude,172,106.4,59.3,6.3,0.642,0.619
7,mechanosignaling,gemini,172,56.2,115.2,0.6,0.328,0.327
8,mechanosignaling,gpt,172,81.5,87.6,2.9,0.482,0.474


## Guardar resultados

In [7]:
salida = comp[["red", "llm", "n_replicas", "TP", "FP", "FN", "recall_media", "recall_sd", "precision_media", "precision_sd",
               "recall_repo", "precision_repo", "dif_recall", "dif_precision", "coincide",
               "verdad_total", "signo_contrario_media", "recall_estricto"]].copy()
salida = salida.merge(resumen_A[["red", "llm", "tabla_identica_a_manuscrito", "celdas_distintas", "conteos_coinciden_recall_tables"]], on=["red", "llm"])
salida.to_csv(SALIDA / "reproduccion_paper_resumen.csv", index=False, encoding="utf-8")
por_replica.to_csv(SALIDA / "reproduccion_paper_por_replica.csv", index=False, encoding="utf-8")
print("Guardado en", SALIDA)

Guardado en C:\Users\lauta\Desktop\Biologia Computacional\Examen Parcial\Benchmarking biochemical networks\experimento\resultados
